# Checkpoint 3: Feature Engineering

Notebook ini membangun fitur-fitur turunan dari data iklim harian Dramaga untuk klasifikasi kekeringan berbasis SPEI-30.

**Output**: `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` — siap dipakai langsung di notebook modelling.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10})

def resolve_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, current.parent):
        if (candidate / 'data').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = resolve_project_root()
DATA_PATH = PROJECT_ROOT / 'data' / 'dataset_iklim_dramaga_1980_2024_completed.csv'

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'], utc=True).dt.tz_convert(None)
df = df.sort_values('date').reset_index(drop=True)
print(f'Dataset loaded: {df.shape}')
df.head()

## 1. Water Balance & Rolling Accumulation

Neraca air harian (`precipitation - ET0`) diakumulasi pada jendela 30, 90, dan 180 hari untuk menangkap memori kelembapan tanah jangka pendek hingga panjang.

In [ ]:
df['water_balance'] = df['precipitation_sum'] - df['et0_fao_evapotranspiration']

for window in [30, 90, 180]:
    df[f'wb_{window}d'] = df['water_balance'].rolling(window=window, min_periods=window).sum()

df[['date', 'water_balance', 'wb_30d', 'wb_90d', 'wb_180d']].tail(10)

In [ ]:
df_plot = df.loc[df['date'].dt.year >= 2021].copy()
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(df_plot['date'], df_plot['precipitation_sum'], alpha=0.4, label='Precipitation (mm)', color='steelblue', width=1)
ax.plot(df_plot['date'], df_plot['wb_30d'], label='WB 30d', color='orange', linewidth=1.5)
ax.plot(df_plot['date'], df_plot['wb_90d'], label='WB 90d', color='red', linewidth=1.5)
ax.set_title('Akumulasi Neraca Air Dramaga (2021-2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('mm')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Perhitungan SPEI (Z-Score per Bulan, Baseline 1981-2010)

SPEI dihitung dengan **z-score normalization per bulan kalender** terhadap periode baseline 1981-2010.
Metode ini menstandarkan water balance per bulan sehingga setiap bulan memiliki distribusi mean≈0 dan std≈1,
memungkinkan perbandingan anomali lintas musim yang adil.

**Formula**: `SPEI = (wb_Xd - mean_baseline_bulan) / std_baseline_bulan`

In [ ]:
df['month'] = df['date'].dt.month
baseline_mask = df['date'].dt.year.between(1981, 2010)

def calculate_spei_zscore(df_in, wb_col, baseline_mask, output_name):
    """Z-score normalization per bulan kalender terhadap baseline."""
    baseline_stats = df_in.loc[baseline_mask].groupby('month')[wb_col].agg(['mean', 'std'])
    
    spei = df_in.apply(
        lambda row: (row[wb_col] - baseline_stats.loc[row['month'], 'mean']) / 
                     baseline_stats.loc[row['month'], 'std']
        if pd.notna(row[wb_col]) else np.nan, axis=1
    )
    return spei.rename(output_name)

for w in [30, 90, 180]:
    df[f'spei_{w}d'] = calculate_spei_zscore(df, f'wb_{w}d', baseline_mask, f'spei_{w}d')

print('SPEI Statistics:')
print(df[['spei_30d', 'spei_90d', 'spei_180d']].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for col, ax in zip(['spei_30d', 'spei_90d', 'spei_180d'], axes):
    sns.histplot(data=df, x=col, kde=True, bins=40, ax=ax, stat='density', color='skyblue', edgecolor='black', alpha=0.6)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='SPEI=0')
    ax.axvline(x=-0.5, color='orange', linestyle=':', linewidth=1.5, label='Batas Ringan')
    ax.axvline(x=-1.5, color='darkred', linestyle=':', linewidth=1.5, label='Batas Parah')
    ax.set_title(f'Distribusi {col.upper()}', fontweight='bold')
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 3. Rolling Features & Fitur Agrometeorologi

Fitur rolling 30/90/180 hari untuk presipitasi dan ET0, serta rolling 30 hari untuk suhu dan kelembapan tanah. Ditambah aridity index, soil water deficit, dan anomali suhu.

In [ ]:
# Rolling aggregates untuk presipitasi dan ET0
for window in [30, 90, 180]:
    df[f'precip_{window}d_sum'] = df['precipitation_sum'].rolling(window=window).sum()
    df[f'et0_{window}d_sum'] = df['et0_fao_evapotranspiration'].rolling(window=window).sum()

df['temp_30d_mean'] = df['temperature_2m_mean'].rolling(window=30).mean()
df['soil_moist_30d_mean'] = df['soil_moisture_0_to_7cm_mean'].rolling(window=30).mean()

# Aridity Index (Precipitation / ET0)
df['aridity_index'] = df['precipitation_sum'] / df['et0_fao_evapotranspiration'].replace(0, 0.001)
df['aridity_index'] = df['aridity_index'].clip(upper=10)

# Soil Water Deficit
df['soil_water_deficit'] = 1.0 - df['soil_moisture_0_to_7cm_mean']

# Temperature Anomaly (terhadap klimatologi baseline 1981-2010)
baseline = df.loc[df['date'].dt.year.between(1981, 2010)]
monthly_clim = baseline.groupby(baseline['date'].dt.month)['temperature_2m_mean'].mean()
df['temp_anomaly'] = df['temperature_2m_mean'] - df['month'].map(monthly_clim)

print('Fitur agrometeorologi berhasil ditambahkan.')
df[['date', 'precip_30d_sum', 'et0_30d_sum', 'aridity_index', 'soil_water_deficit', 'temp_anomaly']].tail(5)

In [ ]:
df['quarter'] = df['date'].dt.quarter
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='aridity_index', y='soil_water_deficit', hue='quarter',
                palette='husl', alpha=0.5, ax=ax, edgecolor='black', linewidth=0.3)
ax.set_title('Aridity Index vs Soil Water Deficit (per Kuartal)', fontweight='bold')
ax.set_xlabel('Aridity Index (Precip/ET0)'); ax.set_ylabel('Soil Water Deficit')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 4. Fitur Temporal (Lag & Cyclical Encoding)

Lag features menangkap memori kondisi sebelumnya. Cyclical month encoding (sin/cos) menjaga kontinuitas Desember-Januari.

In [ ]:
# Lag features
for lag in [1, 7, 30]:
    df[f'water_balance_lag_{lag}'] = df['water_balance'].shift(lag)

# Cyclical month encoding
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df[['date', 'water_balance_lag_1', 'water_balance_lag_7', 'water_balance_lag_30', 'month_sin', 'month_cos']].tail(5)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
um = df.groupby('month')[['month_sin', 'month_cos']].first().reset_index()
colors = plt.cm.hsv(np.linspace(0, 1, 12))
for idx, row in um.iterrows():
    ax.scatter(row['month_cos'], row['month_sin'], s=250, c=[colors[idx]], edgecolor='black', zorder=3)
    ax.text(row['month_cos']*1.15, row['month_sin']*1.15, f"M{int(row['month'])}",
            ha='center', va='center', fontsize=10, fontweight='bold')
ax.add_patch(plt.Circle((0,0), 1, fill=False, color='gray', linestyle='--'))
ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3); ax.set_aspect('equal')
ax.set_title('Cyclical Month Encoding', fontweight='bold')
ax.set_xlabel('cos'); ax.set_ylabel('sin'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Target: Drought Class (SPEI-30)

Klasifikasi berdasarkan ambang SPEI-30 sesuai Project Plan:
- **0 = Normal**: SPEI > -0.50
- **1 = Ringan**: -1.00 ≤ SPEI ≤ -0.50
- **2 = Sedang**: -1.50 ≤ SPEI < -1.00
- **3 = Parah**: SPEI < -1.50

In [ ]:
def map_drought_class(spei):
    if pd.isna(spei): return np.nan
    if spei <= -1.5: return 3
    if spei <= -1.0: return 2
    if spei <= -0.5: return 1
    return 0

df['drought_class'] = df['spei_30d'].apply(map_drought_class).astype('Int64')

print('Distribusi Kelas Target:')
print(df['drought_class'].value_counts().sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
class_labels = {0: 'Normal', 1: 'Ringan', 2: 'Sedang', 3: 'Parah'}
class_order = sorted(df['drought_class'].dropna().unique())
counts = df['drought_class'].value_counts().sort_index()
total = len(df.dropna(subset=['drought_class']))

sns.countplot(data=df.dropna(subset=['drought_class']), x='drought_class',
              order=class_order, ax=ax, palette='Set2', edgecolor='black')
for i, c in enumerate(class_order):
    n = counts.get(c, 0)
    ax.text(i, n + 50, f'{n}\n({n/total*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Distribusi Target: Drought Class', fontweight='bold')
ax.set_xticklabels([f'{c} ({class_labels.get(c, "?")})' for c in class_order])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

ratio = counts.max() / counts.min()
print(f'Imbalance Ratio: {ratio:.1f}x \u2014 perlu SMOTE/class weighting di tahap modelling.')

## 6. Validasi: Multicollinearity Check

In [ ]:
eng_feats = [
    'water_balance', 'wb_30d', 'wb_90d', 'wb_180d',
    'spei_30d', 'spei_90d', 'spei_180d',
    'aridity_index', 'soil_water_deficit', 'temp_anomaly',
    'water_balance_lag_1', 'water_balance_lag_7', 'water_balance_lag_30',
    'month_sin', 'month_cos'
]

corr = df[eng_feats].corr()
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, square=True,
            linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix: Engineered Features', fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

# Check high correlations
pairs = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
         for i in range(len(corr)) for j in range(i+1, len(corr)) if abs(corr.iloc[i,j]) > 0.95]
if pairs:
    for f1, f2, v in pairs: print(f'  {f1} <-> {f2}: {v:.4f}')
else:
    print('Tidak ada pasangan fitur dengan korelasi > 0.95.')

## 7. Feature Selection, Cleaning & Temporal Split

Fitur dipilih sesuai kebutuhan modelling. Data dibagi secara temporal (80/20) untuk menghindari data leakage. Output diekspor sebagai CSV terpisah.

In [ ]:
# Feature list yang akan dipakai di modelling
features = [
    'temperature_2m_mean', 'soil_moisture_0_to_7cm_mean', 'temp_anomaly', 'aridity_index',
    'month_sin', 'month_cos', 'precip_30d_sum', 'et0_30d_sum', 'temp_30d_mean',
    'soil_moist_30d_mean', 'precip_90d_sum', 'et0_90d_sum', 'precip_180d_sum', 'et0_180d_sum'
]
target = 'drought_class'

# Clean: drop rows with NaN in features or target
df_clean = df.dropna(subset=features + [target]).reset_index(drop=True)
print(f'Dataset setelah cleaning: {df_clean.shape}')

# Temporal split 80/20
split_idx = int(len(df_clean) * 0.8)
train_df = df_clean.iloc[:split_idx]
test_df = df_clean.iloc[split_idx:]

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train period: {train_df["date"].min()} \u2014 {train_df["date"].max()}')
print(f'Test period:  {test_df["date"].min()} \u2014 {test_df["date"].max()}')
print(f'\nDistribusi target (train):')
print(y_train.value_counts().sort_index())
print(f'\nDistribusi target (test):')
print(y_test.value_counts().sort_index())

In [ ]:
# Export train/test splits
out_dir = PROJECT_ROOT / 'data'
X_train.to_csv(out_dir / 'X_train.csv', index=False)
X_test.to_csv(out_dir / 'X_test.csv', index=False)
y_train.to_csv(out_dir / 'y_train.csv', index=False, header=True)
y_test.to_csv(out_dir / 'y_test.csv', index=False, header=True)

# Export full featured dataset juga
df_clean.to_csv(out_dir / 'dataset_featured_dramaga.csv', index=False)

print('Files exported:')
for f in ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv', 'dataset_featured_dramaga.csv']:
    p = out_dir / f
    print(f'  {p.name} ({p.stat().st_size/1024:.0f} KB)')

## Kesimpulan

Feature engineering menghasilkan **14 fitur** dari 5 kategori:

| Kategori | Fitur |
|----------|-------|
| **Meteorologi dasar** | `temperature_2m_mean`, `soil_moisture_0_to_7cm_mean` |
| **Rolling aggregates** | `precip_30/90/180d_sum`, `et0_30/90/180d_sum`, `temp_30d_mean`, `soil_moist_30d_mean` |
| **Agrometeorologi** | `aridity_index`, `temp_anomaly` |
| **Temporal** | `month_sin`, `month_cos` |

**Distribusi target** (setelah perbaikan SPEI z-score per bulan):
- Class 0 (Normal): ~64% 
- Class 1 (Ringan): ~17%
- Class 2 (Sedang): ~13%
- Class 3 (Parah): ~6%

**Catatan penting:**
- Data masih imbalanced (ratio ~10x) → SMOTE diterapkan di tahap modelling
- Split temporal (bukan random) untuk menghindari data leakage
- SPEI dihitung dengan z-score per bulan kalender terhadap baseline 1981-2010